In [ ]:
import pandas as pd
import requests
import time
import os
import re

# ================== CONFIG ==================
API_KEY = "LAST_FM_KEY"

INPUT_FILE = "leftover.csv"
CHECKPOINT_FILE = "leftover_check.csv"
FINAL_FILE = "remaining.csv"

TRACK_COLUMN = "name"
ARTIST_COLUMN = "main_artist"
OUTPUT_COLUMN = "genres"

CHECKPOINT_INTERVAL = 100

BAD_TAGS = [
    "seen live", "favorites", "favorite",
    "female vocalists", "male vocalists",
    "90s", "2000s", "2010s",
    "british", "american"
]
BAD_TAGS = set(tag.lower().strip() for tag in BAD_TAGS)

# 🎯 TAG NORMALIZATION / MERGING
TAG_MAPPING = {
    "hip hop": "hip-hop",
    "hiphop": "hip-hop",
    "hip-hop": "hip-hop",
    "rap": "hip-hop",

    "rnb": "r&b",
    "r&b": "r&b",

    "alt rock": "alternative rock",
    "alternative": "alternative rock",

    "indie rock": "indie",
    "indie pop": "indie",

    "electro": "electronic",
    "electronica": "electronic"
}

def normalize_tag(tag):
    tag = tag.lower().strip()
    tag = re.sub(r"\s+", " ", tag)
    return TAG_MAPPING.get(tag, tag)

# ================== FETCH FUNCTIONS ==================
def fetch_track_tags(track, artist, retries=3):
    url = "http://ws.audioscrobbler.com/2.0/"
    params = {
        "method": "track.getTopTags",
        "track": track,
        "artist": artist,
        "api_key": API_KEY,
        "format": "json"
    }
    
    for attempt in range(retries):
        try:
            res = requests.get(url, params=params, timeout=10)
            data = res.json()
            tags = data.get("toptags", {}).get("tag", [])
            if isinstance(tags, list):
                return [(t["name"], int(t.get("count", 0))) for t in tags if "name" in t]
            return []
        except Exception:
            print(f"⚠️ Track retry {attempt+1} failed: {track} - {artist}")
            time.sleep(1)
    return []

def fetch_artist_tags(artist, retries=3):
    url = "http://ws.audioscrobbler.com/2.0/"
    params = {
        "method": "artist.getTopTags",
        "artist": artist,
        "api_key": API_KEY,
        "format": "json"
    }

    for attempt in range(retries):
        try:
            res = requests.get(url, params=params, timeout=10)
            data = res.json()
            tags = data.get("toptags", {}).get("tag", [])
            if isinstance(tags, list):
                return [(t["name"], int(t.get("count", 0))) for t in tags if "name" in t]
            return []
        except Exception:
            print(f"⚠️ Artist retry {attempt+1} failed: {artist}")
            time.sleep(1)
    return []

# ================== LOAD / RESUME ==================
if os.path.exists(CHECKPOINT_FILE):
    print("🔁 Resuming from checkpoint...")
    df = pd.read_csv(CHECKPOINT_FILE)
else:
    print("🆕 Starting fresh...")
    df = pd.read_csv(INPUT_FILE)

    if OUTPUT_COLUMN not in df.columns:
        df[OUTPUT_COLUMN] = None

print("Columns:", df.columns.tolist())

# ================== FIND START INDEX ==================
processed_mask = df[OUTPUT_COLUMN].notna()

if processed_mask.any():
    start_idx = processed_mask[processed_mask].index.max() + 1
else:
    start_idx = 0

print(f"▶️ Starting from index: {start_idx}")

# ================== MAIN LOOP ==================
for idx in range(start_idx, len(df)):

    track = str(df.at[idx, TRACK_COLUMN]).strip()
    artist = str(df.at[idx, ARTIST_COLUMN]).strip()

    if not track or track == "nan" or not artist or artist == "nan":
        continue

    print(f"\n🎵 {idx}: {track} - {artist}")

    raw_tags = fetch_track_tags(track, artist)
    source = "track"

    artist_norm = artist.lower().strip()

    # ================== TRACK TAGS ==================
    cleaned_tags = []

    for tag, count in raw_tags:
        tag_norm = normalize_tag(tag)

        if tag_norm in BAD_TAGS:
            continue
        if tag_norm == artist_norm:
            continue

        cleaned_tags.append(tag_norm)

    cleaned_tags = list(dict.fromkeys(cleaned_tags))

    # ================== FALLBACK ==================
    if not cleaned_tags:
        print("   ⚠️ No usable track tags → using artist fallback")
        raw_artist_tags = fetch_artist_tags(artist)
        source = "artist"

        processed = []

        for tag, count in raw_artist_tags:
            tag_norm = normalize_tag(tag)

            if tag_norm in BAD_TAGS:
                continue
            if tag_norm == artist_norm:
                continue

            processed.append((tag_norm, count))

        # sort by count
        processed.sort(key=lambda x: x[1], reverse=True)

        # take top 3
        cleaned_tags = [tag for tag, _ in processed[:3]]

        cleaned_tags = list(dict.fromkeys(cleaned_tags))

    # ================== PRINT ==================
    if cleaned_tags:
        print(f"   → ({source}) Genres: {', '.join(cleaned_tags)}")
    else:
        print("   → Genres: NONE (after filtering)")

    df.at[idx, OUTPUT_COLUMN] = ",".join(cleaned_tags)

    time.sleep(0.3)

    # ================== CHECKPOINT ==================
    if idx % CHECKPOINT_INTERVAL == 0 and idx != 0:
        df.to_csv(CHECKPOINT_FILE, index=False)
        print(f"💾 Checkpoint saved at index {idx}")

# ================== FINAL SAVE ==================
df.to_csv(FINAL_FILE, index=False)
print("\n✅ Finished! File saved as:", FINAL_FILE)

🆕 Starting fresh...
Columns: ['name', 'main_artist', 'genres']
▶️ Starting from index: 0

🎵 0: Got 2 Believe in Magic - Version 1 - Side A
   ⚠️ No usable track tags → using artist fallback
   → (artist) Genres: free jazz, jazz, opm

🎵 1: Ya Soshla S Uma - t.A.T.u.
   ⚠️ No usable track tags → using artist fallback
   → (artist) Genres: pop, russian, electronic

🎵 2: Jealous - Nina
   ⚠️ No usable track tags → using artist fallback
   → (artist) Genres: synthwave, eurodance, pop

🎵 3: Bluer Than Blue - Remastered 2001 - Michael Johnson
   ⚠️ No usable track tags → using artist fallback
   → (artist) Genres: pop, country, singer-songwriter

🎵 4: All The Things She Said - t.A.T.u.
   → (track) Genres: pop, dance, russian, tatu, electronic, 00s, rock, pop rock, lesbian

🎵 5: ProvÃ³came - Chayanne
   ⚠️ No usable track tags → using artist fallback
   → (artist) Genres: latin, latin pop, spanish

🎵 6: Otra Vez - VÃ­ctor GarcÃ­a
   ⚠️ No usable track tags → using artist fallback
   → Genres: